<a href="https://colab.research.google.com/github/Nurdaylight/Study/blob/main/Text_scraper_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install playwright
!python -m playwright install chromium
!python -m playwright install-deps chromium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 18.7 MB/s eta 0:00:00
(node:549) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 62.1s167.3 MiB [] 0% 35.2s167.3 MiB [] 0% 24.7s167.3 MiB [] 0% 23.0s167.3 MiB [] 0% 19.2s167.3 MiB [] 0% 12.5s167.3 MiB [] 1% 8.0s167.3 MiB [] 2% 6.5s167.3 MiB [] 2% 6.4s167.3 MiB [] 2% 6.5s167.3 MiB [] 3% 5.5s167.3 MiB [] 4% 4.7s167.3 MiB [] 4% 4.3s167.3 MiB [] 5% 4.1s167.3 MiB [] 6% 3.9s167.3 MiB [] 6% 3.7s167.3 MiB [] 7% 3.6s167.3 MiB [] 8% 3.5s167.3 MiB [] 8% 3.3s167.3 MiB [] 9% 3.2s167.3 MiB [] 10% 3.1s167.3 MiB [] 11% 2.9s167.3 MiB [] 11% 2.8s167.3 MiB [] 12% 2.8s167.3 MiB [] 13% 2.7s167.3 MiB [] 14% 2.6s167.3 MiB [] 15% 2.6s167.3 MiB [] 16% 2.6s167.3 MiB [] 16% 2.5s167.3 M

In [4]:
from bs4 import BeautifulSoup

# read file containing HTML
with open("sample_data/rune HTMLS.txt", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

# extract all links
links = [a["href"] for a in soup.find_all("a", href=True)]


In [ ]:
import asyncio
import os
from playwright.async_api import async_playwright

def clean_lines(text: str) -> str:
    lines = [ln.strip() for ln in text.splitlines()]
    return "\n".join([ln for ln in lines if ln])

async def scrape_chapters_from_links(links, delay_sec: float = 1.0):
    os.makedirs("readings", exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--no-zygote",
                "--single-process",
            ],
        )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            locale="en-US",
        )

        page = await context.new_page()
        await page.set_extra_http_headers({
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://novelbin.com/",
        })

        for idx, url in enumerate(links):
            print(f"\nScraping ({idx+1}/{len(links)}): {url}")

            await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_selector("#chr-content", timeout=30000)

            title = (await page.locator("h2").first.inner_text()).strip()
            raw = await page.locator("#chr-content").inner_text()

            text = clean_lines(raw)

            filename = f"readings/chapter_{links[idx][-10:-1]}.txt"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(title + "\n\n" + text)

            print(f"Saved → {filename}")
            await asyncio.sleep(delay_sec)

        await context.close()
        await browser.close()
i=250
await scrape_chapters_from_links(links[i:i+1], delay_sec=3)


Scraping (1/1): https://novelbin.com/b/return-of-the-runebound-professor/chapter-250-casino
Saved → readings/chapter_250-casin.txt
